# День 5 — Fine-tuning модели

Цель: дообучить DistilBERT на IMDB и сравнить с baseline из Дня 4.
Токенизатор — из Дня 1, данные — `data/imdb_train.csv`.

## Задача 1: Dataset класс

`SentimentDataset` находится в `sentiment_lab.dataset`: он токенизирует
текст с `padding='max_length'`, обрезает до `max_length` и возвращает
`input_ids`, `attention_mask` и `labels` (torch.long).

In [ ]:
from torch.utils.data import DataLoader

from sentiment_lab.dataset import SentimentDataset
from sentiment_lab.tokenization import load_tokenizer

model_name = "distilbert-base-uncased"
tokenizer = load_tokenizer(model_name)
sample = SentimentDataset(["Great movie!"], [1], tokenizer, max_length=16)[0]
print({key: tuple(value.shape) for key, value in sample.items()})

## Задача 2: загрузка и подготовка данных

Для скорости на CPU берём 500 отзывов; `N_SAMPLES` можно увеличить до 5000.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/imdb_train.csv").sample(n=500, random_state=42)
texts = df["text"].tolist()
labels = df["label"].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)
print(f'Train: {len(train_dataset)}, val: {len(val_dataset)}')

## Задача 3: модель для классификации и DataLoaders

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = len(set(labels))
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
print(f'num_labels: {num_labels}, batches: {len(train_loader)} / {len(val_loader)}')

## Задача 4: настройка обучения

В transformers 5.x `AdamW` больше не экспортируется — используется
`torch.optim.AdamW`.

In [ ]:
from torch.optim import AdamW

from sentiment_lab.training import get_device

device = get_device()
optimizer = AdamW(model.parameters(), lr=2e-5)
model.to(device)
print(f'Device: {device}')

## Задачи 5–7: epoch, оценка и обучение

`train_epoch` и `evaluate` лежат в `sentiment_lab.training`; `train_model`
вызывает их в цикле и печатает метрики каждой эпохи.

In [ ]:
from sentiment_lab.training import train_model

result = train_model(model, train_loader, val_loader, num_epochs=3, optimizer=optimizer)
print(f'Final Val F1: {result.val_f1:.4f}')

## Задача 8: сохранение модели и метрик

Модель — в `models/fine_tuned_model`, метрики — в `outputs/fine_tuned_results.txt`
(обе папки в `.gitignore`).

In [ ]:
from sentiment_lab.training import save_finetuned

model_dir, results_path = save_finetuned(model, tokenizer, result)
print(f'Модель: {model_dir}')
print(f'Метрики: {results_path}')